# DVC on a single zarr store (single-GPU, all knobs explicit)

Run `correlate_multi_gpu` against one deformation entry of one zarr store and visualize the recovered displacement field. Every parameter is pinned as a named constant in the **Knobs** cell — nothing is auto-tuned or silently filled in from a config file.

**Footgun, read me first.** The transduced synchrotron stores under `D:\jannik\synchrotron-data\transduced\` store their ground-truth flow in **push_forward** convention, but the registered store profile (`bone_screw_synchrotron_v1`) declares **pull_back**. Running with the wrong convention silently inflates the error report ~50× (per-axis MAE ≈ 2 × |u|). The `FLOW_CONVENTION` knob below overrides the profile per-run.

**GPU mode.** This notebook pins `DEVICE_IDS = [0]` so dispatch takes its single-process in-process fast path — no `multiprocessing.spawn`, no notebook-vs-`__main__` pickling drama. Multi-GPU runs belong in `scripts/run_e2e_zarr.py`.

**I/O note.** `D:` is uncached on this host (~0.27 GB/s); a full `(960, 1280, 1280)` float32 volume pair is ~50 s of pure read. `DRY_SHAPE` crops to a centered subblock for fast smoke runs.

In [ ]:
from __future__ import annotations

import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pyvista as pv

from mamba_dvc.core.ncc import NCCMode, NCCNormalization
from mamba_dvc.gpu.dispatch import correlate_multi_gpu
from mamba_dvc.io.dataset import NO_MASK, DvcDataset
from mamba_dvc.io.manifest import StoreManifest
from mamba_dvc.types import POIStatus
from mamba_dvc.viz.backend import plotter
from mamba_dvc.viz.field import render_error_glyphs, render_field, render_field_lattice
from mamba_dvc.viz.volume import render_volume

pv.set_jupyter_backend("trame")

## Knobs

Every parameter pinned as a named constant. Change one thing at a time and re-run from here.

In [ ]:
!dir C:\Users\jstebani\Desktop\dvc-data-storage

In [ ]:
# --- store + entry selection ---------------------------------------------
STORE_PATH = Path(r"C:\Users\jstebani\Desktop\dvc-data-storage\103L_Mg5Gd_4w_000.zarr")
DEFORMATION = "fs104"        # one of ds.list_real() / ds.list_synthetic(), set after Cell 4
MASK: str | None = "mask_fill"    # str = named mask; None = profile default; sentinel below for skip
SKIP_MASK = False            # True overrides MASK and passes NO_MASK to load_pair
STRICT = True                # flip to False for 5L_PEEK_4w_000.zarr (fs402 corrupted upstream)

# --- flow-convention override (the manual-via-string knob) ---------------
# Transduced stores: "push_forward". The profile says "pull_back". Wrong
# value inflates the error report ~50x (per-axis MAE ~ 2x |u|).
FLOW_CONVENTION: str = "push_forward"

# --- volume slicing ------------------------------------------------------
# Full volume is (960, 1280, 1280) on these stores. (192, 256, 256) is the
# fast smoke setting; None reads the full volume (~50 s of pure I/O from D:).
DRY_SHAPE: tuple[int, int, int] | None = None

# --- correlate_multi_gpu knobs ------------------------------------------
DEVICE_IDS: list[int] = [0]           # single-process fast path; >1 needs a .py entrypoint
WINDOW: int = 64                      # subvolume size in voxels (DRY_SHAPE allows smaller W)
OVERLAP: float = 0.5                  # POI stride = (1 - OVERLAP) * WINDOW
MASK_THRESHOLD: float = 0.9           # min valid-voxel fraction to admit a POI
TUKEY_ALPHA: float | None = None      # None = mode default (0.0 for linear, 0.25 for cyclic)
SEARCH_RADIUS: int | None = None      # None = WINDOW // 2; larger = wider unambiguous lag
BATCH_SIZE: int = 64                  # explicit int -> dispatch is fully side-effect-free
EPS: float = 1e-12                    # NCC denominator floor
NCC_MODE: NCCMode = NCCMode.LINEAR
NCC_NORMALIZATION: NCCNormalization = NCCNormalization.OVERLAP

## Open the store with the flow-convention override

Start from a discovered sidecar manifest if one exists (rare for the transduced stores), then patch `synthetic.flow.convention` so `GroundTruthField` decodes with the right sign. This is the exact pattern `scripts/run_e2e_zarr.py:_apply_flow_convention_override` uses, so the notebook and the harness agree.

In [ ]:
base_manifest = StoreManifest.discover(STORE_PATH) or StoreManifest()
patched_flow = replace(base_manifest.synthetic.flow, convention=FLOW_CONVENTION)  # type: ignore[arg-type]
patched_synthetic = replace(base_manifest.synthetic, flow=patched_flow)
manifest = replace(base_manifest, synthetic=patched_synthetic)

ds = DvcDataset.open(STORE_PATH, manifest=manifest, strict=STRICT)

print(f"store:        {STORE_PATH.name}")
print(f"profile:      {ds.profile.name}")
print(f"volume_shape: {ds.volume_shape}")
print(f"spacing:      {ds.spacing}")
print(f"masks:        {ds.list_masks()}")
print(f"real:         {ds.list_real()}")
print(f"synthetic:    {ds.list_synthetic()}")
print(f"broken:       {ds.list_broken()}")
print(f"flow conv.:   {manifest.synthetic.flow.convention}")

## Materialize one (reference, deformed, mask, gt_field) bundle

`load_pair` reads from disk and coerces to the canonical dtypes. With `DRY_SHAPE` set, all four arrays are cropped to the same centered subblock so the mask and GT field stay aligned with the volumes.

In [ ]:
mask_selector = NO_MASK if SKIP_MASK else MASK

t0 = time.perf_counter()
pair = ds.load_pair(DEFORMATION, mask=mask_selector, dry_shape=DRY_SHAPE)
load_seconds = time.perf_counter() - t0

print(f"loaded in {load_seconds:.2f}s")
print(f"  reference  {pair.reference.shape} {pair.reference.dtype}  ({pair.reference.nbytes / 1e6:.1f} MB)")
print(f"  deformed   {pair.deformed.shape} {pair.deformed.dtype}  ({pair.deformed.nbytes / 1e6:.1f} MB)")
if pair.mask is not None:
    fg = float(pair.mask.mean())
    print(f"  mask       {pair.mask.shape} {pair.mask.dtype}  foreground fraction = {fg:.3f}")
else:
    print("  mask       (none)")
print(f"  gt_field   {'present' if pair.gt_field is not None else 'absent (real entry)'}")
print(f"  kind:      {pair.kind}")

## Run the GPU correlator

`correlate_multi_gpu` with `device_ids=[0]` skips spawn and uploads the volumes to device 0 in-process. All FFT + peakfit work runs on the GPU; grid construction, mask admission, and the outlier test stay on the host. The result is bit-identical (within float32 noise) to the multi-GPU path.

In [ ]:
t0 = time.perf_counter()
result = correlate_multi_gpu(
    pair.reference,
    pair.deformed,
    mask=pair.mask,
    device_ids=DEVICE_IDS,
    window=WINDOW,
    overlap=OVERLAP,
    mask_threshold=MASK_THRESHOLD,
    tukey_alpha=TUKEY_ALPHA,
    search_radius=SEARCH_RADIUS,
    batch_size=BATCH_SIZE,
    eps=EPS,
    ncc_mode=NCC_MODE,
    ncc_normalization=NCC_NORMALIZATION,
)
compute_seconds = time.perf_counter() - t0

print(f"correlate_multi_gpu: {compute_seconds:.2f}s")
print(f"  lattice    {result.grid_shape}  (= {result.positions.shape[0]} POIs)")
print(f"  spacing    {result.spacing} voxels")
print(f"  window     {result.window} voxels")
print()
print("  POI status:")
for status in POIStatus:
    n = int(np.count_nonzero(result.status == int(status)))
    if n:
        pct = 100.0 * n / result.status.size
        print(f"    {status.name:14s} {n:6d}  ({pct:5.1f}%)")

valid = result.valid
if valid.any():
    mag = np.linalg.norm(result.displacements[valid], axis=1)
    conf = result.confidence[valid]
    print()
    print("  |displacement| (voxels, valid only):")
    print(f"    p50={np.percentile(mag, 50):.3f}  p90={np.percentile(mag, 90):.3f}  "
          f"p99={np.percentile(mag, 99):.3f}  max={mag.max():.3f}")
    print("  confidence (valid only):")
    print(f"    p50={np.percentile(conf, 50):.3f}  p10={np.percentile(conf, 10):.3f}  "
          f"min={conf.min():.3f}")

In [ ]:
result.displacements.shape

## Visualize the recovered field — displacement glyphs

Arrow glyphs at every valid POI, colored by displacement magnitude, drawn over the reference volume. `GLYPH_FACTOR` is a *visual* multiplier; it does not change the underlying numbers. A reasonable rule of thumb: pick the factor so `factor × p99(|u|)` is ~5× the lattice spacing — otherwise glyphs are either invisible or overdraw each other.

In [ ]:
GLYPH_FACTOR = 3.0     # visual amplification only
GLYPH_STRIDE: int | None = None  # e.g. 2 to thin the cloud on dense lattices

with plotter(window_size=(1000, 750)) as p:
    p.add_text(f"recovered displacement — {DEFORMATION}", font_size=10)
    render_volume(p, pair.reference, mask=pair.mask, spacing=pair.spacing, cmap="bone")
    render_field(
        p,
        result,
        spacing=pair.spacing,
        only_valid=True,
        scale="magnitude",
        factor=GLYPH_FACTOR,
        cmap="viridis",
        stride=GLYPH_STRIDE,
        scalar_bar_kw={"title": "||u|| [voxels]"},
    )
    p.show()

## Visualize the recovered field — magnitude lattice

Same data, different idiom. The POI lattice is colored as a structured grid by `||u||`, no glyphs. Useful when the *direction* of the displacement is not what you want to communicate (e.g., "where is the screw pulling the tissue hardest?") or when the glyph cloud above is too dense to read.

In [ ]:
with plotter(window_size=(1000, 750)) as p:
    p.add_text(f"||u|| over the POI lattice — {DEFORMATION}", font_size=10)
    render_field_lattice(
        p,
        result,
        spacing=pair.spacing,
        scalar="magnitude",
        cmap="viridis",
        opacity=1.0,
    )
    p.show()

## (Optional) Error vs ground truth

Skipped for real entries (no flow on disk). For synthetic entries the GT field is sampled at the recovered POI centers; the error glyphs are colored by `||recovered − truth||`. If the per-axis MAE looks like ~2× the per-axis mean displacement, `FLOW_CONVENTION` is set wrong — flip it and re-run.

In [ ]:
if pair.gt_field is None:
    print(f"entry {DEFORMATION!r} is a real deformation; no GT field, skipping error analysis.")
else:
    truth = pair.gt_field(result.positions)
    valid = result.valid
    err = (result.displacements - truth)[valid]
    mag_err = np.linalg.norm(err, axis=1)

    print(f"error vs GT on {int(valid.sum())} valid POIs:")
    print(f"  MAE_z {np.mean(np.abs(err[:, 0])):.4f}  "
          f"MAE_y {np.mean(np.abs(err[:, 1])):.4f}  "
          f"MAE_x {np.mean(np.abs(err[:, 2])):.4f}")
    print(f"  RMSE  {float(np.sqrt((err**2).mean())):.4f}")
    print(f"  p50   {np.percentile(mag_err, 50):.4f}  "
          f"p95 {np.percentile(mag_err, 95):.4f}  "
          f"max {mag_err.max():.4f}")

    ERROR_GLYPH_FACTOR = 40.0  # errors are decades smaller than displacements
    with plotter(window_size=(1000, 750)) as p:
        p.add_text(f"error glyphs (recovered − truth) — {DEFORMATION}", font_size=10)
        render_error_glyphs(
            p,
            result,
            truth=truth,
            spacing=pair.spacing,
            only_valid=True,
            factor=ERROR_GLYPH_FACTOR,
            cmap="magma",
            scalar_bar_kw={"title": "|error| [voxels]"},
        )
        p.show()